# The Clustering Layer That Public Health Surveillance Is Missing

> **How Embedding Choice Affects Signal Detectability in Clinical Free-Text**

This notebook reproduces the full experiment from the Towards Data Science article.
It is self-contained and Colab-ready — all you need is the MTSamples CSV and
(optionally) an OpenAI API key for Method 3.

---

## Table of contents
1. [Problem setup](#1-problem-setup)
2. [Data loading](#2-data-loading)
3. [Three embedding methods](#3-three-embedding-methods)
4. [Clustering and evaluation](#4-clustering-and-evaluation)
5. [Signal injection experiment](#5-signal-injection-experiment)
6. [Visualisations](#6-visualisations)

## 0 · Environment setup

Run this cell once to install all dependencies (Colab or fresh venv).

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys

packages = [
    "pandas", "numpy", "scikit-learn",
    "sentence-transformers", "umap-learn",
    "fast-hdbscan", "matplotlib", "seaborn",
    "openai", "tqdm", "joblib",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("All packages installed.")

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
    "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD",
]
print("Imports OK.")

---

## 1 · Problem setup

Health surveillance systems ingest a continuous stream of clinical free-text:
triage notes, chief complaints, discharge summaries.  The classic pipeline
converts that text to vectors and groups similar records via clustering.

The question we ask: **does the choice of text embedding method affect whether
a real epidemiological signal — say, an emerging cluster of patients with
identical symptoms — becomes *detectable*?**

We operationalise this with the **Signal Injection Experiment**:

1. Take a labelled corpus of clinical notes (MTSamples).
2. Cluster with three embedding methods of increasing semantic sophistication.
3. Inject 50 synthetic records, all describing the same clinical event
   (*acute vision loss after medication*), written in 10 different surface forms.
4. Measure what fraction of those 50 records land in the *same* cluster.

If TF-IDF scatters them across clusters → the signal is **missed**.  
If a semantic method groups them together → the signal is **detected**.

---

## 2 · Data loading

We use the **MTSamples** dataset (4,999 medical transcription records, CC0 licensed).
The key fields are:
- `description` — a 1–3 sentence clinical summary (our input text)
- `medical_specialty` — the specialty category (our ground-truth label)

We keep only the **top 10 specialties** by record count to keep the experiment focused.

In [ ]:
# ── Data loader ───────────────────────────────────────────────────────────────

DATA_PATH = Path("../data/mtsamples.csv")   # adjust if running from repo root
TOP_N = 10

def load_mtsamples(path: Path = DATA_PATH, top_n: int = TOP_N) -> pd.DataFrame:
    """Load MTSamples, filter to top-N specialties, return clean DataFrame."""
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found at {path}.\n"
            "Download from https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions"
        )

    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df = df[["description", "medical_specialty"]].copy()
    df.dropna(inplace=True)
    df["description"] = df["description"].str.strip()
    df["medical_specialty"] = df["medical_specialty"].str.strip()
    df = df[df["description"].str.len() > 0]

    top_specs = df["medical_specialty"].value_counts().head(top_n).index.tolist()
    df = df[df["medical_specialty"].isin(top_specs)].copy()

    label_map = {name: i for i, name in enumerate(sorted(top_specs))}
    df["specialty"] = df["medical_specialty"]
    df["specialty_id"] = df["specialty"].map(label_map)
    df.drop(columns=["medical_specialty"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df[["description", "specialty", "specialty_id"]]


df = load_mtsamples()
texts = df["description"].tolist()
true_labels = df["specialty_id"].to_numpy()
specialty_names = sorted(df["specialty"].unique())

print(f"Records: {len(df):,}  |  Specialties: {len(specialty_names)}")
df["specialty"].value_counts()

---

## 3 · Three embedding methods

We compute embeddings three ways, in increasing order of semantic richness.

### Method 1 — TF-IDF + LSA (baseline)
Classic bag-of-words with bigrams, reduced to 100 dimensions via
Truncated SVD (Latent Semantic Analysis).  Fast, no GPU, no API.  
Weakness: two sentences can mean the same thing and score zero similarity.

In [ ]:
# ── TF-IDF + LSA ──────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

CACHE_TFIDF = RESULTS_DIR / "embeddings_tfidf.npy"

def embed_tfidf(texts, cache_suffix="tfidf"):
    cache = RESULTS_DIR / f"embeddings_{cache_suffix}.npy"
    if cache.exists():
        print(f"[cache] Loading TF-IDF embeddings from {cache}")
        return np.load(cache)
    t0 = time.perf_counter()
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5_000, ngram_range=(1, 2), sublinear_tf=True)),
        ("svd",   TruncatedSVD(n_components=100, random_state=RANDOM_STATE)),
    ])
    arr = pipe.fit_transform(texts).astype(np.float32)
    np.save(cache, arr)
    print(f"[TF-IDF] {arr.shape} in {time.perf_counter()-t0:.1f}s — saved to {cache}")
    return arr

emb_tfidf = embed_tfidf(texts)
print("TF-IDF embedding shape:", emb_tfidf.shape)

### Method 2 — Sentence-transformers (all-MiniLM-L6-v2)

A 22M-parameter transformer fine-tuned on 1B sentence pairs for semantic
similarity.  Produces 384-dimensional unit-normalised vectors.  Runs entirely
locally — no API key needed.

In [ ]:
# ── MiniLM-L6-v2 ──────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

def embed_minilm(texts, cache_suffix="minilm"):
    cache = RESULTS_DIR / f"embeddings_{cache_suffix}.npy"
    if cache.exists():
        print(f"[cache] Loading MiniLM embeddings from {cache}")
        return np.load(cache)
    t0 = time.perf_counter()
    model = SentenceTransformer("all-MiniLM-L6-v2")
    arr = model.encode(
        texts, batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32)
    np.save(cache, arr)
    print(f"[MiniLM] {arr.shape} in {time.perf_counter()-t0:.1f}s — saved to {cache}")
    return arr

emb_minilm = embed_minilm(texts)
print("MiniLM embedding shape:", emb_minilm.shape)

### Method 3 — OpenAI text-embedding-3-small

OpenAI's latest embedding model.  Produces 1536-dimensional vectors.  
Requires `OPENAI_API_KEY` in environment.  Skipped gracefully if absent.

In [ ]:
# ── OpenAI text-embedding-3-small ─────────────────────────────────────────────
# Set your key: os.environ["OPENAI_API_KEY"] = "sk-..."  ← do NOT commit!

def embed_openai(texts, cache_suffix="openai", chunk_size=100, max_retries=5):
    """Returns embedding array, or None if OPENAI_API_KEY is not set."""
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        print("[OpenAI] OPENAI_API_KEY not set — skipping.")
        return None

    cache = RESULTS_DIR / f"embeddings_{cache_suffix}.npy"
    if cache.exists():
        print(f"[cache] Loading OpenAI embeddings from {cache}")
        return np.load(cache)

    from openai import OpenAI, RateLimitError
    from tqdm import tqdm

    client = OpenAI(api_key=api_key)
    t0 = time.perf_counter()
    chunks = [texts[i:i+chunk_size] for i in range(0, len(texts), chunk_size)]
    all_embs = []

    for chunk in tqdm(chunks, desc="[OpenAI] batches"):
        for attempt in range(max_retries):
            try:
                resp = client.embeddings.create(model="text-embedding-3-small", input=chunk)
                all_embs.extend([np.array(e.embedding, dtype=np.float32) for e in resp.data])
                break
            except RateLimitError:
                wait = 2 ** attempt
                print(f"  Rate limit — waiting {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError("Max retries exceeded.")

    arr = np.vstack(all_embs)
    np.save(cache, arr)
    print(f"[OpenAI] {arr.shape} in {time.perf_counter()-t0:.1f}s — saved to {cache}")
    return arr

emb_openai = embed_openai(texts)
if emb_openai is not None:
    print("OpenAI embedding shape:", emb_openai.shape)
else:
    print("OpenAI embeddings not available — continuing with Methods 1 & 2.")

---

## 4 · Clustering and evaluation

- **TF-IDF** → **K-means** (k=10).  We know there are 10 specialties, so we
  give K-means the benefit of the doubt.
- **Semantic methods** → **HDBSCAN**.  Density-based; does not require
  specifying k; handles irregular cluster shapes.  Noise points (label −1)
  are excluded from metric calculations.

**Metrics:**
- *ARI* — how well cluster assignments align with ground-truth specialties
- *Silhouette* — intrinsic cluster quality (no labels needed), computed on
  2-D UMAP projections

In [ ]:
# ── Clustering ────────────────────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from umap import UMAP

try:
    from fast_hdbscan import HDBSCAN
except ImportError:
    raise ImportError("Install fast-hdbscan: pip install fast-hdbscan")

HDBSCAN_PARAMS = dict(min_cluster_size=15, min_samples=5)

def cluster_kmeans(emb, k=10):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    return km.fit_predict(emb).astype(np.int32)

def cluster_hdbscan(emb):
    return HDBSCAN(**HDBSCAN_PARAMS).fit_predict(emb).astype(np.int32)

def umap_2d(emb):
    return UMAP(n_components=2, random_state=RANDOM_STATE).fit_transform(emb)

def compute_ari(labels, true):
    mask = labels != -1
    return adjusted_rand_score(true[mask], labels[mask]) if mask.sum() > 0 else 0.0

def compute_sil(xy, labels, n=2000):
    mask = labels != -1
    X, y = xy[mask], labels[mask]
    if len(np.unique(y)) < 2: return float("nan")
    if len(X) > n:
        idx = np.random.default_rng(RANDOM_STATE).choice(len(X), n, replace=False)
        X, y = X[idx], y[idx]
    return silhouette_score(X, y)

# ── Run ───────────────────────────────────────────────────────────────────────
embedding_map = {"tfidf": emb_tfidf, "minilm": emb_minilm}
if emb_openai is not None:
    embedding_map["openai"] = emb_openai

cluster_fn_map = {
    "tfidf":  cluster_kmeans,
    "minilm": cluster_hdbscan,
    "openai": cluster_hdbscan,
}

cluster_labels = {}
umap_2d_map    = {}
results        = {}

for key, emb in embedding_map.items():
    print(f"\n── {key} ──")
    cluster_labels[key] = cluster_fn_map[key](emb)
    print(f"  Clusters: {(np.unique(cluster_labels[key]) >= 0).sum()}  "
          f"Noise: {(cluster_labels[key] == -1).sum()}")
    print(f"  UMAP …")
    umap_2d_map[key] = umap_2d(emb)
    ari = compute_ari(cluster_labels[key], true_labels)
    sil = compute_sil(umap_2d_map[key], cluster_labels[key])
    results[key] = {"ari": ari, "silhouette": sil}
    print(f"  ARI={ari:.4f}  Silhouette={sil:.4f}")

---

## 5 · Signal injection experiment

This is the novel contribution of the article.

We inject 50 synthetic records — all describing *acute vision disturbance
after medication* — written in 10 different surface forms (5 copies each).  
These are embedded using each method and the full dataset is re-clustered.

**Recovery Rate** = fraction of the 50 that land in the modal cluster.  
**Cluster Purity** = fraction of that modal cluster that is synthetic.

- TF-IDF: expected to scatter (different word choices → different vectors)
- Semantic: expected to cluster tightly (same meaning → similar vectors)

In [ ]:
# ── Synthetic outbreak records ────────────────────────────────────────────────

OUTBREAK_PHRASES = [
    "sudden vision loss after taking medication",
    "patient reports blurry eyesight post-dose",
    "acute visual disturbance following drug administration",
    "can't see clearly since starting the medication",
    "eyes went blurry after first pill",
    "visual acuity decreased after medication initiation",
    "sight problems beginning after prescription started",
    "went blind in one eye after taking the drug",
    "difficulty seeing following pharmacological treatment",
    "ocular symptoms appeared post-medication",
]

RECORDS_PER_PHRASE = 5
injected_texts = [p for p in OUTBREAK_PHRASES for _ in range(RECORDS_PER_PHRASE)]
N_INJECTED = len(injected_texts)
print(f"Synthetic outbreak records: {N_INJECTED}")
print("\nSample surface forms:")
for p in OUTBREAK_PHRASES[:3]:
    print(f"  • {p}")

In [ ]:
# ── Embed + cluster + measure ─────────────────────────────────────────────────

embed_fn_map = {
    "tfidf":  embed_tfidf,
    "minilm": embed_minilm,
    "openai": embed_openai,
}

umap_combined = {}   # UMAP projections of base + injected (for visualisation)
n_base = len(texts)

for key, emb in embedding_map.items():
    print(f"\n── Signal injection: {key} ──")
    inj_emb = embed_fn_map[key](injected_texts, cache_suffix=f"{key}_injected")
    if inj_emb is None:
        print("  Skipped (no embedding available).")
        results[key].update({"recovery_rate": float("nan"),
                              "cluster_purity": float("nan")})
        continue

    combined = np.vstack([emb, inj_emb])
    all_labels = cluster_fn_map[key](combined)
    inj_labels = all_labels[n_base:]

    non_noise = inj_labels[inj_labels != -1]
    if len(non_noise) == 0:
        print("  All injected records are noise — recovery = 0")
        results[key].update({"recovery_rate": 0.0, "cluster_purity": 0.0})
        continue

    vals, cnts = np.unique(non_noise, return_counts=True)
    modal = int(vals[cnts.argmax()])
    modal_cnt = int(cnts.max())
    recovery = modal_cnt / N_INJECTED
    modal_total = int((all_labels == modal).sum())
    purity = modal_cnt / modal_total if modal_total > 0 else 0.0

    print(f"  Recovery rate:  {recovery:.1%}  ({modal_cnt}/{N_INJECTED})")
    print(f"  Cluster purity: {purity:.1%}  ({modal_cnt}/{modal_total})")
    results[key].update({"recovery_rate": recovery, "cluster_purity": purity})

    # UMAP for visualisation
    umap_combined[key] = umap_2d(combined)

In [ ]:
# ── Print results table ───────────────────────────────────────────────────────

METHOD_LABELS = {
    "tfidf":  "TF-IDF + K-means",
    "minilm": "MiniLM + HDBSCAN",
    "openai": "OpenAI + HDBSCAN",
}

rows = []
for k, v in results.items():
    rows.append({
        "Method": METHOD_LABELS.get(k, k),
        "ARI": f"{v.get('ari', float('nan')):.4f}",
        "Silhouette": f"{v.get('silhouette', float('nan')):.4f}",
        "Recovery Rate": f"{v.get('recovery_rate', float('nan')):.4f}",
        "Cluster Purity": f"{v.get('cluster_purity', float('nan')):.4f}",
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(RESULTS_DIR / "summary.csv", index=False)

---

## 6 · Visualisations

Three figures, all saved to `results/`.

In [ ]:
# ── Figure 1: UMAP coloured by specialty ─────────────────────────────────────

unique_ids = np.unique(true_labels)
color_map = {sid: PALETTE[i % len(PALETTE)] for i, sid in enumerate(unique_ids)}

methods_present = [k for k in METHOD_LABELS if k in umap_2d_map]
fig, axes = plt.subplots(1, len(methods_present), figsize=(5*len(methods_present), 4.5))
if len(methods_present) == 1: axes = [axes]

for ax, key in zip(axes, methods_present):
    xy = umap_2d_map[key]
    for sid in unique_ids:
        mask = true_labels == sid
        ax.scatter(xy[mask,0], xy[mask,1], c=color_map[sid],
                   s=8, alpha=0.6, linewidths=0)
    ax.set_title(METHOD_LABELS[key], fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")

patches = [mpatches.Patch(color=color_map[sid], label=specialty_names[sid])
           for sid in unique_ids]
fig.legend(handles=patches, loc="lower center", ncol=5,
           fontsize=7, frameon=False, bbox_to_anchor=(0.5,-0.05))
fig.suptitle("UMAP — Coloured by Medical Specialty",
             fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "umap_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved → results/umap_comparison.png")

In [ ]:
# ── Figure 2: Signal injection UMAP ──────────────────────────────────────────

if umap_combined:
    methods_inj = [k for k in METHOD_LABELS if k in umap_combined]
    fig, axes = plt.subplots(1, len(methods_inj), figsize=(5*len(methods_inj), 4.5))
    if len(methods_inj) == 1: axes = [axes]

    for ax, key in zip(axes, methods_inj):
        xy = umap_combined[key]
        ax.scatter(xy[:n_base,0], xy[:n_base,1],
                   c="#CCCCCC", s=6, alpha=0.35, linewidths=0, label="Real records")
        ax.scatter(xy[n_base:,0], xy[n_base:,1],
                   c="#E63946", s=30, alpha=0.85, linewidths=0,
                   label="Injected (outbreak)", zorder=5)
        ax.set_title(METHOD_LABELS[key], fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")

    legend_patches = [
        mpatches.Patch(color="#CCCCCC", label="Real records"),
        mpatches.Patch(color="#E63946", label="Injected outbreak records"),
    ]
    fig.legend(handles=legend_patches, loc="lower center",
               ncol=2, fontsize=8, frameon=False, bbox_to_anchor=(0.5,-0.04))
    fig.suptitle("Signal Injection — Where Outbreak Records Land",
                 fontsize=11, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / "signal_injection_umap.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved → results/signal_injection_umap.png")
else:
    print("No injection UMAP data available.")

In [ ]:
# ── Figure 3: Summary metrics bar chart ──────────────────────────────────────

methods_bar = [k for k in METHOD_LABELS if k in results]
labels_bar  = [METHOD_LABELS[m] for m in methods_bar]

ari_vals = [results[m].get("ari",           float("nan")) for m in methods_bar]
sil_vals = [results[m].get("silhouette",    float("nan")) for m in methods_bar]
rec_vals = [results[m].get("recovery_rate", float("nan")) for m in methods_bar]

x = np.arange(len(methods_bar))
w = 0.25

fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x-w,  ari_vals, w, label="ARI",               color=PALETTE[0], alpha=0.85)
b2 = ax.bar(x,    sil_vals, w, label="Silhouette",         color=PALETTE[2], alpha=0.85)
b3 = ax.bar(x+w,  rec_vals, w, label="Signal Recovery Rate", color="#E63946",  alpha=0.85)

for bars in (b1, b2, b3):
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f"{h:.2f}",
                    ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(labels_bar, fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Clustering Quality and Signal Detection by Embedding Method",
             fontweight="bold", fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.axhline(1.0, color="#999999", linewidth=0.6, linestyle="--")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "summary_metrics.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved → results/summary_metrics.png")

---

## Summary

The experiment confirms the central argument of the article:

- **TF-IDF** clusters by keyword overlap — records describing the same
  phenomenon in different words are scattered, making the injected outbreak
  effectively invisible.
- **Semantic embeddings** (MiniLM / OpenAI) understand meaning, not just
  surface form, and concentrate the injected records into a single tight
  cluster — enabling outbreak detection.

The gap in Signal Recovery Rate between the baseline and semantic methods
is a direct, quantifiable measure of the surveillance blind spot that
lexical methods create.

---

*Dataset: MTSamples, CC0 1.0 Universal Public Domain Dedication.*  
*Reproducible experiment for the Towards Data Science article.*